<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>

<br>

# <font color="#76b900">**Notebook 8 [Assessment]:** RAG Evaluation</font>

<br>

Welcome to the last notebook of the course! In the previous notebook, you integrated a vector store solution into a RAG pipeline! In this notebook, you will take that same pipeline and evaluate it using numerical RAG evaluation techniques incorporating LLM-as-a-Judge metrics!

<br>

### **Learning Objectives:**

- Learn how to integrate the techniques from prior notebooks to numerically approximate the goodness of your RAG pipeline.

- **Final Exercice**: ***By working through this notebook in the Course Environment,* you will be able to submit the coding component of the course!**

<br>

### **Questions To Think About:**

- As you go along, remember what our metrics actually represent. Should our pipeline pass these objectives? Is our judge LLM sufficient for evaluating the pipeline? Does a particular metric even matter for our use case?
- If we left the vectorstore-as-a-memory component in our chain, do you think it would still pass the evaluation? Additionally, is the evaluation useful for assessing vectorstore-as-a-memory performance? 

<br>

### **Notebook Source:**

- This notebook is part of a larger [**NVIDIA Deep Learning Institute**](https://www.nvidia.com/en-us/training/) course titled [**Building RAG Agents with LLMs**](https://www.nvidia.com/en-sg/training/instructor-led-workshops/building-rag-agents-with-llms/). If sharing this material, please give credit and link back to the original course.

<br>

### **Environment Setup:**

In [1]:
# %pip install -q langchain langchain-nvidia-ai-endpoints gradio rich
# %pip install -q arxiv pymupdf faiss-cpu ragas

## If you encounter a typing-extensions issue, restart your runtime and try again
# from langchain_nvidia_ai_endpoints import ChatNVIDIA
# ChatNVIDIA.get_available_models()

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
norm_style = Style(bold=True)
pprint = partial(console.print, style=base_style)
pprint2 = partial(console.print, style=norm_style)

from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

# NVIDIAEmbeddings.get_available_models()
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")

# ChatNVIDIA.get_available_models(base_url="http://llm_client:9000/v1")
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x7b-instruct-v0.1")

----

<br>

## **Part 1:** Pre-Release Evaluation

In our previous notebook, we successfully combined several concepts to create a document chatbot with the aim of responsive and informative interactions. However, the diversity of user interactions necessitates comprehensive testing to truly understand the chatbot's performance. Thorough testing in varied scenarios is crucial to ensure that the system is not only robust and versatile but also aligns with user and provider expectations.

After defining your chatbot's roles and implementing the necessary features, evaluating it becomes a multi-stage process:

- **Typical Use Inspection:** Start by testing scenarios most relevant to your use case. See if your chatbot can reliably navigate discussions with limited human intervention.

    - Additionally, identify limitations or compartments that should be redirected to a human for inspection/supervision (i.e., human swap-in to confirm transactions or perform sensitive navigation) and implement those options.

- **Edge Case Inspection:** Explore the boundaries of typical use, identifying how the chatbot handles less common but plausible scenarios.

    - Before any public release, assess critical boundary conditions that could pose liability risks, such as the potential generation of inappropriate content.

    - Implement well-tested guardrails on all outputs (and possibly inputs) to limit undesired interactions and redirect users into predictable conversation flows.

- **Progressive Rollout:** Rolling out your model to a limited audience (first internal, then [A/B](https://en.wikipedia.org/wiki/A/B_testing)) and implement analytics features like usage analytics dashboards and feedback avenues (flag/like/dislike/etc).

Of these three steps, the first two can be done by a small team or an individual and should be iterated on as part of the development process. Unfortunately, this needs to be done frequently and can be prone to human error. **Luckily for us, LLMs can be used to help out with LLM-as-a-Judge formulations!**

*(Yeah, this probably isn't surprising by now. LLMs being strong is why this course is here...).*

----

<br>

## **Part 2:** LLM-as-a-Judge Formulation

In the realm of conversational AI, using LLMs as evaluators or 'judges' has emerged as a useful approach for configurable automatic testing of natural language task performance:

- An LLM can simulate a range of interaction scenarios and generate synthetic data, allowing an evaluation developer to generate targeted inputs to eliciting a range of behaviors from your chatbot.

- The chatbot's correspondence/retrieval on the synthetic data can be evaluated or parsed by an LLM and a consistent output format such as "Pass"/"Fail", similarity, or extraction can be enforced.

- Many such results can be aggregated and a metric can be derived which explains something like "% of passing evaluations", "average number of relevant details from the sources", "average cosine similarity", etc.

This idea of using LLMs to test out and quantify chatbot quality, known as [**"LLM-as-a-Judge,"**](https://arxiv.org/abs/2306.05685) allows for easy test specifications that align closely with human judgment and can be fine-tuned and replicated at scale.

**There are several popular frameworks for off-the-shelf judge formulations including:**
- [**RAGAs (short for RAG Assessment)**](https://docs.ragas.io/en/stable/), which offers a suite of great starting points for your own evaluation efforts.
- [**LangChain Evaluators**](https://python.langchain.com/v0.1/docs/guides/productionization/evaluation/), which are similar first-party options with many implicitly-constructible agents.

Instead of using the chains as-is, we will instead expand on the ideas and evaluate our system with a more custom solution.

----

<br>

## **Part 3: [Assessment Prep]** Pairwise Evaluator

The following exercise will flesh out a custom implementation of a simplified [LangChain Pairwise String Evaluator](https://python.langchain.com/v0.1/docs/guides/productionization/evaluation/comparison/pairwise_string/). 

**To prepare for our RAG chain evaluation, we will need to:**

- Pull in our document index (the one we saved in the previous notebook).
- Recreate our RAG pipeline of choice.

**We will specifically be implementing a judge formulation with the following steps:**

- Sample the RAG agent document pool to find two document chunks.
- Use those two document chunks to generate a synthetic "baseline" question-answer pair.
- Use the RAG agent to generate its own answer.
- Use a judge LLM to compare the two responses while grounding the synthetic generation as "ground-truth correct."

**The chain should be a simple but powerful process that tests for the following objective:**

> ***Does my RAG chain outperform a narrow chatbot with limited document access.***





**This will be the system used for the final evaluation!** To see how this system is integrated into the autograder, please check out the implementation in [`frontend/server_app.py`](frontend/server_app.py).

<br>

### **Task 1:** Pull In Your Document Retrieval Index

For this exercise, you will pull in the `docstore_index` file you created as part of your earlier notebook. The following cell should be able to load in the store as-is.

In [2]:
## Make sure you have docstore_index.tgz in your working directory
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS

# embedder = NVIDIAEmbeddings(model="nvidia/embed-qa-4", truncate="END")

!tar xzvf docstore_index.tgz
docstore = FAISS.load_local("docstore_index", embedder, allow_dangerous_deserialization=True)
docs = list(docstore.docstore._dict.values())

def format_chunk(doc):
    return (
        f"Paper: {doc.metadata.get('Title', 'unknown')}"
        f"\n\nSummary: {doc.metadata.get('Summary', 'unknown')}"
        f"\n\nPage Body: {doc.page_content}"
    )

## This printout just confirms that your store has been retrieved
pprint(f"Constructed aggregate docstore with {len(docstore.docstore._dict)} chunks")
pprint(f"Sample Chunk:")
print(format_chunk(docs[len(docs)//2]))

# The code confirms that the vector store has been successfully loaded and displays:
# The total number of document chunks.
# A sample document chunk with its title, summary, and content.

tar: Error opening archive: Failed to open 'docstore_index.tgz'


Constructed aggregate docstore with 369 chunks

Sample Chunk:

Paper: Mistral 7B

Summary: We introduce Mistral 7B v0.1, a 7-billion-parameter language model engineered
for superior performance and efficiency. Mistral 7B outperforms Llama 2 13B
across all evaluated benchmarks, and Llama 1 34B in reasoning, mathematics, and
code generation. Our model leverages grouped-query attention (GQA) for faster
inference, coupled with sliding window attention (SWA) to effectively handle
sequences of arbitrary length with a reduced inference cost. We also provide a
model fine-tuned to follow instructions, Mistral 7B -- Instruct, that surpasses
the Llama 2 13B -- Chat model both on human and automated benchmarks. Our
models are released under the Apache 2.0 license.

Page Body: . We measure performance on a wide variety of tasks categorized as follow:\n\u2022 Commonsense Reasoning (0-shot): Hellaswag [28], Winogrande [21], PIQA [4], SIQA [22],\nOpenbookQA [19], ARC-Easy, ARC-Challenge [9], CommonsenseQA [24]\n\u2022 World Knowledge (5-shot): NaturalQuestions [1

<br>

### **Task 2: [Exercise]** Pull In Your RAG Chain

Now that we have our index, we can recreate the RAG agent from the previous notebook! 

**Key Modifications:**
- To keep things simple, feel free to disregard the vectorstore-as-a-memory component. Incorporating it will require some more overhead and will make the exercise a bit more complicated.

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.runnables.passthrough import RunnableAssign
from langchain.document_transformers import LongContextReorder

from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

from functools import partial
from operator import itemgetter

import gradio as gr

#####################################################################

# NVIDIAEmbeddings.get_available_models()
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-v1", truncate="END")

# ChatNVIDIA.get_available_models()
instruct_llm = ChatNVIDIA(
    model="meta/llama3-8b-instruct",
    temperature=0.0,
    top_p=1.0,
)

llm = instruct_llm | StrOutputParser()

#####################################################################

def docs2str(docs, title="Document"):
    """Useful utility for making chunks into context string. Optional, but useful"""
    out_str = ""
    for doc in docs:
        doc_name = getattr(doc, 'metadata', {}).get('Title', title)
        if doc_name: out_str += f"[Quote from {doc_name}] "
        out_str += getattr(doc, 'page_content', str(doc)) + "\n"
    return out_str

chat_prompt = ChatPromptTemplate.from_template(
    "You are a document chatbot. Help the user as they ask questions about documents."
    " User messaged just asked you a question: {input}\n\n"
    " The following information may be useful for your response: "
    " Document Retrieval:\n{context}\n\n"
    " (Answer only from retrieval. Only cite sources that are used. Make your response conversational)"
    "\n\nUser Question: {input}"
)

def output_puller(inputs):
    """"Output generator. Useful if your chain returns a dictionary with key 'output'"""
    if isinstance(inputs, dict):
        inputs = [inputs]
    for token in inputs:
        if token.get('output'):
            yield token.get('output')

#####################################################################
## TODO: Pull in your desired RAG Chain. Memory not necessary

## Chain 1 Specs: "Hello World" -> retrieval_chain 
##   -> {'input': <str>, 'context' : <str>}
long_reorder = RunnableLambda(LongContextReorder().transform_documents)  ## GIVEN
context_getter = (
    lambda x: docs2str(
        long_reorder.invoke(
            docstore.similarity_search(x['input'], k=3)
        )
    )
)
retrieval_chain = {'input' : (lambda x: x)} | RunnableAssign({'context' : context_getter})

## Chain 2 Specs: retrieval_chain -> generator_chain 
##   -> {"output" : <str>, ...} -> output_puller
generator_chain = (
    chat_prompt
    | llm
)
generator_chain = {'output' : generator_chain} | RunnableLambda(output_puller)  ## GIVEN

## END TODO
#####################################################################

rag_chain = retrieval_chain | generator_chain

# pprint(rag_chain.invoke("Tell me something interesting!"))
for token in rag_chain.stream("Tell me something interesting about RAG!"):
    print(token, end="")

RAG! Retrieval-Augmented Generation is a fascinating topic. Did you know that RAG is designed to enhance the trustworthiness of Large Language Models (LLMs) by dynamically retrieving information from external knowledge databases? This approach has been widely adopted in real-world applications, including ChatGPT, Microsoft Bing Chat, Perplexity AI, and Google Search AI (Chen et al., 2024b; Gao et al., 2023; Lewis et al., 2020).

However, recent incidents have revealed critical weaknesses in these systems, including inconsistent Google Search AI results and dangerous malicious code injections (BBC, 2024; rocky, 2024). This highlights the importance of addressing the fundamental challenge of corpus poisoning attacks, which can compromise the accuracy of RAG systems.

Despite these challenges, RAG has shown promising results in various applications. For instance, the RAG-Sequence model uses the same retrieved document to predict each target token, while the RAG-Token model can predict eac

<br>

### **Step 3:** Generating Synthetic Question-Answer Pairs

In this section, we can implement the first few part of our evaluation routine:

- **Sample the RAG agent document pool to find two document chunks.**
- **Use those two document chunks to generate a synthetic "baseline" question-answer pair.**
- Use the RAG agent to generate its own answer.
- Use a judge LLM to compare the two responses while grounding the synthetic generation as "ground-truth correct."

The chain should be a simple but powerful process that tests for the following objective:

> Does my RAG chain outperform a narrow chatbot with limited document access?

In [16]:
import random

num_questions = 3
synth_questions = []
synth_answers = []

simple_prompt = ChatPromptTemplate.from_messages([('system', '{system}'), ('user', 'INPUT: {input}')])

for i in range(num_questions):
    doc1, doc2 = random.sample(docs, 2)
    sys_msg = (
        "Use the documents provided by the user to generate an interesting question-answer pair."
        " Try to use both documents if possible, and rely more on the document bodies than the summary."
        " Use the format:\nQuestion: (good question, 1-3 sentences, detailed)\n\nAnswer: (answer derived from the documents)"
        " DO NOT SAY: \"Here is an interesting question pair\" or similar. FOLLOW FORMAT!"
    )
    usr_msg = (
        f"Document1: {format_chunk(doc1)}\n\n"
        f"Document2: {format_chunk(doc2)}"
    )

    qa_pair = (simple_prompt | llm).invoke({'system': sys_msg, 'input': usr_msg})
    synth_questions += [qa_pair.split('\n\n')[0]]
    synth_answers += [qa_pair.split('\n\n')[1]]
    pprint2(f"QA Pair {i+1}")
    pprint2(synth_questions[-1])
    pprint(synth_answers[-1])
    print()

QA Pair 1

Question: How do retrieval-augmented generation models compare to traditional parametric seq2seq models in terms of
language generation quality and factual accuracy?

Answer: According to the paper "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks", 
retrieval-augmented generation (RAG) models generate more specific, diverse, and factual language than a 
state-of-the-art parametric-only seq2seq baseline. This is evident from the results shown in Table 2, where RAG 
models outperform the parametric seq2seq model in terms of generation and classification test scores. Additionally,
the paper highlights that RAG models can generate more accurate and informative language by leveraging the 
knowledge stored in the non-parametric memory.

QA Pair 2

Question: How do large language models, like those used in BERT, interact with external knowledge sources and 
discrete reasoning modules in the MRKL system?

Answer: In the MRKL system, large language models, such as those used in BERT, interact with external knowledge 
sources and discrete reasoning modules through a specialized neural net called the router. The router is 
responsible for extracting discrete parameters required by the module from the text, which must be done rigorously 
through training. For example, in the case of Jurassic-X, the router was trained to extract basic arithmetic 
operations from language descriptions. This allows the MRKL system to combine the strengths of large language 
models with the capabilities of external knowledge sources and discrete reasoning modules.

QA Pair 3

Question: How can we effectively evaluate large language models (LLMs) in open-ended tasks, such as multi-turn 
dialogues, to ensure their alignment with human preferences?

Answer: We can use strong LLMs as judges to evaluate these models on more open-ended questions, and introduce 
benchmarks like MT-bench and Chatbot Arena to measure the agreement between LLM judges and human preferences. Our 
results show that strong LLM judges like GPT-4 can match both controlled and crowdsourced human preferences well, 
achieving over 80% agreement, the same level of agreement between humans.

<br>

### **Step 4:** Answer The Synthetic Questions

In this section, we can implement the third part of our evaluation routine:

- Sample the RAG agent document pool to find two document chunks.
- Use those two document chunks to generate a synthetic "baseline" question-answer pair.
- **Use the RAG agent to generate its own answer.**
- Use a judge LLM to compare the two responses while grounding the synthetic generation as "ground-truth correct."

The chain should be a simple but powerful process that tests for the following objective:

> Does my RAG chain outperform a narrow chatbot with limited document access?

In [17]:
## TODO: Generate some synthetic answers to the questions above.
##   Try to use the same syntax as the cell above
rag_answers = []
for i, q in enumerate(synth_questions):
    ## TODO: Compute the RAG Answer
    rag_answer = rag_chain.invoke(q)
    rag_answers += [rag_answer]
    pprint2(f"QA Pair {i+1}", q, "", sep="\n")
    pprint(f"RAG Answer: {rag_answer}", "", sep='\n')


QA Pair 1
Question: How do retrieval-augmented generation models compare to traditional parametric seq2seq models in terms of
language generation quality and factual accuracy?

RAG Answer: So, you're wondering how retrieval-augmented generation models stack up against traditional parametric 
seq2seq models when it comes to language generation quality and factual accuracy.

According to the research, retrieval-augmented generation models (RAG) tend to outperform traditional parametric 
seq2seq models in terms of language generation quality and factual accuracy. In fact, a study found that RAG models
generate more specific, diverse, and factual language than a state-of-the-art parametric-only seq2seq baseline [1].

In particular, RAG models were shown to excel in knowledge-intensive generation tasks, such as generating responses
to questions and fact verification. For example, in the MS-MARCO and Jeopardy question generation tasks, RAG models
generated responses that were more factual, specific, and diverse than a BART baseline [1]. Additionally, RAG 
models achieved results within 4.3% of state-of-the-art pipeline models on the FEVER fact verification task [2].

One reason for this improved performance is that RAG models can leverage external knowledge sources, such as 
Wikipedia, to inform their generation. This allows them to produce more accurate and informative responses. In 
contrast, traditional parametric seq2seq models rely solely on their internal parameters to generate text.

Overall, the research suggests that retrieval-augmented generation models are a promising approach for improving 
language generation quality and factual accuracy, especially in knowledge-intensive tasks.

References:

[1] Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks

[2] FEVER: Fact Verification for Natural Language Processing

QA Pair 2
Question: How do large language models, like those used in BERT, interact with external knowledge sources and 
discrete reasoning modules in the MRKL system?

RAG Answer: So, according to the MRKL Systems paper, large language models like those used in BERT are part of a 
modular, neuro-symbolic architecture that combines them with external knowledge sources and discrete reasoning 
modules.

In this architecture, the large language models are referred to as "experts" and are used to process natural 
language inputs. The paper mentions that these experts can be neural, such as general-purpose huge language models 
like BERT, or smaller, specialized LMs.

When an input is received, a router module determines which expert can best respond to the input and routes the 
input to that expert. This expert then processes the input and generates an output, which can be the final output 
of the MRKL system or be routed to another expert for further processing.

The paper also mentions that the MRKL system can integrate external knowledge sources, such as APIs, to provide 
up-to-date information and access to proprietary databases and other information sources.

In terms of discrete reasoning modules, the paper mentions that these can be symbolic, such as a math calculator or
a currency converter, and can be used to provide explanations for the MRKL system's output.

Overall, the MRKL system is designed to be a flexible and modular architecture that can combine the strengths of 
large language models like BERT with the benefits of external knowledge sources and discrete reasoning modules.

Source: MRKL Systems paper, "A modular, neuro-symbolic architecture that combines large language models, external 
knowledge sources and discrete reasoning"

QA Pair 3
Question: How can we effectively evaluate large language models (LLMs) in open-ended tasks, such as multi-turn 
dialogues, to ensure their alignment with human preferences?

RAG Answer: Evaluating large language models (LLMs) in open-ended tasks like multi-turn dialogues is a crucial step
in ensuring their alignment with human preferences. According to a study, "Judging LLM-as-a-Judge with MT-Bench and
Chatbot Arena" [1], a robust and scalable automated method is needed to evaluate LLM alignment with human 
preferences. The study introduces two benchmarks, MT-bench and Chatbot Arena, which use human ratings as the 
primary evaluation metric.

MT-bench is a series of open-ended questions that evaluate a chatbot's multi-turn conversational and 
instruction-following ability, which are critical elements for human preference. The study reveals that strong LLMs
can achieve an agreement rate of over 80%, on par with the level of agreement among human experts, establishing a 
foundation for an LLM-based evaluation framework.

To effectively evaluate LLMs in open-ended tasks, the study suggests combining existing capability-based benchmarks
with preference-based benchmarks using LLM-as-a-judge. This hybrid evaluation framework can swiftly and 
automatically evaluate both the core capabilities and human alignment of models.

In addition, the study provides a sample of multi-turn questions in MT-bench, which can be used as a starting point
for future studies. The questions are categorized into different topics, such as writing, and include sample 
questions like composing an engaging travel blog post or rewriting a previous response.

Overall, evaluating LLMs in open-ended tasks requires a robust and scalable automated method that takes into 
account human preferences. The study provides a foundation for developing such a method and highlights the 
importance of combining capability-based and preference-based benchmarks.

References:
[1] Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena

<br>

### **Step 5:** Implement A Human Preference Metric

In this section, we can implement the fourth part of our evaluation routine:

- Sample the RAG agent document pool to find two document chunks.
- Use those two document chunks to generate a synthetic "baseline" question-answer pair.
- Use the RAG agent to generate its own answer.
- **Use a judge LLM to compare the two responses while grounding the synthetic generation as "ground-truth correct."**

The chain should be a simple but powerful process that tests for the following objective:

> Does my RAG chain outperform a narrow chatbot with limited document access?

In [19]:
## TODO: Adapt this prompt for whichever LLM you're actually interested in using. 
## If it's llama, maybe system message would be good?
eval_prompt = ChatPromptTemplate.from_template("""
You are an impartial and precise evaluator that compares two answers to the same question.

## Task
Given a Question and two Answers, decide which answer is more accurate, complete, and faithful to the question.

- "Answer 1" is the reference (ground truth).
- "Answer 2" is the candidate produced by a system.

## Scoring Rules
- Score 1 if Answer 2 is worse, less accurate, or incomplete compared to Answer 1.
- Score 2 if Answer 2 is equally good or better than Answer 1.

The decision must be strictly factual — do not reward verbosity or style.
Assume all necessary context is in the Question and Answers.

## Output Format
Respond only in the following format:

[Score] Justification

Where:
- Score is 1 or 2.
- Justification is a short factual explanation (max 25 words).

## Example
Input:
Question: What is the capital of France?
Answer 1: The capital of France is Paris.
Answer 2: Paris is the capital of France.

Output:
[2] Both answers are factually identical.

Now evaluate the following:

{qa_trio}

Remember:
- Do not include explanations or any text outside the requested format.
- Do not use markdown, JSON, or code fences.
""")

pref_score = []

trio_gen = zip(synth_questions, synth_answers, rag_answers)
for i, (q, a_synth, a_rag) in enumerate(trio_gen):
    pprint2(f"Set {i+1}\n\nQuestion: {q}\n\n")

    qa_trio = f"Question: {q}\n\nAnswer 1 (Ground Truth): {a_synth}\n\n Answer 2 (New Answer): {a_rag}"
    pref_score += [(eval_prompt | llm).invoke({'qa_trio': qa_trio})]
    pprint(f"Synth Answer: {a_synth}\n\n")
    pprint(f"RAG Answer: {a_rag}\n\n")
    pprint2(f"Synth Evaluation: {pref_score[-1]}\n\n")

Set 1

Question: Question: How do retrieval-augmented generation models compare to traditional parametric seq2seq models 
in terms of language generation quality and factual accuracy?

Synth Answer: Answer: According to the paper "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks", 
retrieval-augmented generation (RAG) models generate more specific, diverse, and factual language than a 
state-of-the-art parametric-only seq2seq baseline. This is evident from the results shown in Table 2, where RAG 
models outperform the parametric seq2seq model in terms of generation and classification test scores. Additionally,
the paper highlights that RAG models can generate more accurate and informative language by leveraging the 
knowledge stored in the non-parametric memory.

RAG Answer: So, you're wondering how retrieval-augmented generation models stack up against traditional parametric 
seq2seq models when it comes to language generation quality and factual accuracy.

According to the research, retrieval-augmented generation models (RAG) tend to outperform traditional parametric 
seq2seq models in terms of language generation quality and factual accuracy. In fact, a study found that RAG models
generate more specific, diverse, and factual language than a state-of-the-art parametric-only seq2seq baseline [1].

In particular, RAG models were shown to excel in knowledge-intensive generation tasks, such as generating responses
to questions and fact verification. For example, in the MS-MARCO and Jeopardy question generation tasks, RAG models
generated responses that were more factual, specific, and diverse than a BART baseline [1]. Additionally, RAG 
models achieved results within 4.3% of state-of-the-art pipeline models on the FEVER fact verification task [2].

One reason for this improved performance is that RAG models can leverage external knowledge sources, such as 
Wikipedia, to inform their generation. This allows them to produce more accurate and informative responses. In 
contrast, traditional parametric seq2seq models rely solely on their internal parameters to generate text.

Overall, the research suggests that retrieval-augmented generation models are a promising approach for improving 
language generation quality and factual accuracy, especially in knowledge-intensive tasks.

References:

[1] Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks

[2] FEVER: Fact Verification for Natural Language Processing

Synth Evaluation: [2] Both answers provide similar information, with Answer 2 being more verbose but factually 
identical to Answer 1.

Set 2

Question: Question: How do large language models, like those used in BERT, interact with external knowledge sources
and discrete reasoning modules in the MRKL system?

...

Synth Evaluation: [1] Answer 2 lacks specific details and technical accuracy compared to Answer 1, which provides a
more precise explanation of the MRKL system's architecture and interaction between components.

Set 3

Question: Question: How can we effectively evaluate large language models (LLMs) in open-ended tasks, such as 
multi-turn dialogues, to ensure their alignment with human preferences?

Synth Answer: Answer: We can use strong LLMs as judges to evaluate these models on more open-ended questions, and 
introduce benchmarks like MT-bench and Chatbot Arena to measure the agreement between LLM judges and human 
preferences. Our results show that strong LLM judges like GPT-4 can match both controlled and crowdsourced human 
preferences well, achieving over 80% agreement, the same level of agreement between humans.

RAG Answer: Evaluating large language models (LLMs) in open-ended tasks like multi-turn dialogues is a crucial step
in ensuring their alignment with human preferences. According to a study, "Judging LLM-as-a-Judge with MT-Bench and
Chatbot Arena" [1], a robust and scalable automated method is needed to evaluate LLM alignment with human 
preferences. The study introduces two benchmarks, MT-bench and Chatbot Arena, which use human ratings as the 
primary evaluation metric.

MT-bench is a series of open-ended questions that evaluate a chatbot's multi-turn conversational and 
instruction-following ability, which are critical elements for human preference. The study reveals that strong LLMs
can achieve an agreement rate of over 80%, on par with the level of agreement among human experts, establishing a 
foundation for an LLM-based evaluation framework.

To effectively evaluate LLMs in open-ended tasks, the study suggests combining existing capability-based benchmarks
with preference-based benchmarks using LLM-as-a-judge. This hybrid evaluation framework can swiftly and 
automatically evaluate both the core capabilities and human alignment of models.

In addition, the study provides a sample of multi-turn questions in MT-bench, which can be used as a starting point
for future studies. The questions are categorized into different topics, such as writing, and include sample 
questions like composing an engaging travel blog post or rewriting a previous response.

Overall, evaluating LLMs in open-ended tasks requires a robust and scalable automated method that takes into 
account human preferences. The study provides a foundation for developing such a method and highlights the 
importance of combining capability-based and preference-based benchmarks.

References:
[1] Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena

Synth Evaluation: [2] Both answers provide similar information, including the introduction of benchmarks, agreement
rates, and a hybrid evaluation framework.

<br>

**Congratulations! We now have an LLM system that reasons about our pipeline and tries to evaluate it!** Now that we have some judge results, we can simply aggregate the results and see how often our formulation was according to an LLM:

In [20]:
pref_score = sum(("[2]" in score) for score in pref_score) / len(pref_score)
print(f"Preference Score: {pref_score}")

Preference Score: 0.6666666666666666


In [17]:

from langserve import RemoteRunnable
from langchain_core.output_parsers import StrOutputParser

llm = RemoteRunnable("http://0.0.0.0:9012/basic_chat/") | StrOutputParser()
for token in llm.stream("Hello World! How is it going?"):
    print(token, end='')


Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to assist you. How can I help you today? Is there something you would like to know or a problem you need help with?

In [34]:
retrieval_chain9012 = RemoteRunnable("http://0.0.0.0:9012/retriever/") | StrOutputParser()
generator_chain9012 = RemoteRunnable("http://0.0.0.0:9012/generator/")
rag_chain9012 = retrieval_chain9012 | generator_chain9012



In [ ]:
for token in retrieval_chain9012.stream("Tell me something interesting about RAG"):
    print(token, end='')


----

<br>

## **Part 4:** Advanced Formulations

The exercise above was meant to prepare you for the final assessment of the course and showcased a simple but effective evaluator chain. The objective and implementation details were provided for you, and the logic for using it probably makes sense now that you've seen it in action. 

With that being said, this metric was merely a product of us specifying:
- **What kind of behavior is important for our pipeline to have?**
- **What do we need to do in order to exhibit and evaluate this behavior?**

From these two questions, we could have come up with plenty of other evaluation metrics that could have assessed different attributes, incorporated different evaluator chain techniques, and even required different pipeline organization strategies. Though far from an exhaustive list, some common formulations you will likely come across may include:

- **Style Evaluation:** Some evaluation formulations can be as simple as "let me ask some questions and see if the output feels desirable." This might be used to see whether a chatbot "acts like it's supposed to" based on a description provided to a judge LLM. We're using quotations since this kind of assessment can reasonably be achieved with nothing but prompt engineering and a while loop.

- **Ground-Truth Evaluation:** In our chain, we used synthetic generation to create some random questions and answers using a sampling strategy, but in reality you may actually have some representative questions and answers that you need your chatbot to consistently get right! In this case, a modification of the exercise chain above should be implemented and closely monitored as you develop your pipelines.

- **Retrieval/Augmentation Evaluation:** This course made many assumptions about what kinds of preprocessing and prompting steps would be good for your pipelines, and much of this was determined by experimentation. Factors such as document preprocessing, chunking strategies, model selection, and prompt specification all played important roles, so creating metrics to validate these decisions may be of interest. This kind of metric might require your pipeline to output your context chunks or may even rely solely on embedding similarity comparisons, so keep this in mind when trying to implement a chain that works with multiple evaluation strategies. Consider the [**RagasEvaluatorChain**](https://docs.ragas.io/en/stable/howtos/integrations/langchain.html) abstraction as a decent starting point for making an custom generalizable evaluation routine. 

- **Trajectory Evaluation:** Using more advanced agent formulations, you can implement multiple-query strategies that assume the presence of conversational memory. With this, you can implement an evaluation agent which can:
    - Ask a series of questions in order to evaluate how well the agent is able to adapt and cater to the scenario. This kind of system generally considers a series of correspondence and aims to tease out and evaluate a "trajectory" of how the agent navigated the conversation. The [**LangChain Trajectory Evaluators documentation**](https://python.langchain.com/v0.1/docs/guides/productionization/evaluation/trajectory/) is a good starting point.
    - Alternatively, you could also implement an evaluation agent that tries to achieve objectives by interacting with the chatbot. Such an agent can output whether they were able to navigate to their solution in a natural manner, and can even be used to generate a report about the percieved performance. The [**LangChain Agents documentation**](https://python.langchain.com/v0.1/docs/modules/agents/) is a good starting point!

<br>

At the end of the day, just make sure to use the tools you have at your disposal appropriately. By this point in the course, you should already be well-acquainted with the LLM core value propositions: **They're powerful, scalable, predictable, controllable, and orchestratable... but will act unpredictably when you just expect them to work by default.** Assess your needs, formulate and validate your pipelines, give enough information, and add as much control as you can to make your system work consistently, efficiently, and effectively.

----

<br>

## **Part 5: [Assessment]** Evaluating For Credit

Welcome to the last exercise of the course! Hopefully you've enjoyed the material and are ready to actually get credit for these notebooks! For this part:

- **Make sure you're in the course environment**
- **Make sure `docstore_index/` has been uploaded to the course environment...**
    - **...and contains [at least one Arxiv paper](https://arxiv.org/search/advanced) which has been updated recently.**
- **Make sure you don't have some old session of [`09_langserve.ipynb`](09_langserve.ipynb) already occupying the port. Your assessment requires you to implement the new `/retriever` and `/generator` endpoints!!**

**Objective:** On launch, [**`frontend/frontend_block.py`**](frontend/frontend_block.py) had several lines of code which trigger the course pass condition. Your objective is to invoke that series of commands by using your pipeline to pass the **Evaluation** check! Recall [`09_langserve.ipynb`](09_langserve.ipynb) and use it as a starting example! As a recommendation, consider duplicating it so that you can keep the original as an authoritative reference. 

**Once Finished:** While your course environment is still open, please navigate back to your course environment launcher area and click the **"Assess Task"** button! After that, you're all done!

In [ ]:
%%js
var url = 'http://'+window.location.host+':8090';
element.innerHTML = '<a style="color:green;" target="_blank" href='+url+'><h1>< Link To Gradio Frontend ></h1></a>';

----

<br>

## <font color="#76b900">**Congratulations On Completing The Course**</font>

Hopefully this course was not only exciting and challenging, but also adequately prepared you for work on the cutting edge of LLM and RAG system development! Going forward, you should have the skills necessary to tackle industry-level challenges and explore RAG deployment with open-source models and frameworks.

**Some NVIDIA-specific releases related to this that you may find interesting include:**
- [**NVIDIA NIM**](https://www.nvidia.com/en-us/ai/), which offers microservice spinup routines that can be deployed on local compute.
- [**TensorRT-LLM**](https://github.com/NVIDIA/TensorRT-LLM) is the current recommended framework for deploying GPU-accelerated LLM model engines in production settings.
- [**NVIDIA's Generative AI Examples Repo**](https://github.com/NVIDIA/GenerativeAIExamples), which includes the current canonical microservice example application and will be updated with new resources as new production workflows get released.
- [**The Knowledge-Based Chatbot Technical Brief**](https://resources.nvidia.com/en-us-generative-ai-chatbot-workflow/knowledge-base-chatbot-technical-brief) which discusses additional publicly-accessible details on productionalizing RAG systems.

**Additionally, some key topics you may be interested in delving more into include:**
- [**LlamaIndex**](https://www.llamaindex.ai/), which has strong components that can augment and occasionally improve upon the LangChain RAG features.
- [**LangSmith**](https://docs.smith.langchain.com/), an upcoming agent productionalization service offered by LangChain.
- [**Gradio**](https://www.gradio.app/), though touched on in the course, has many more interface options which will be worth investigating. For inspiration, consider checking out [**HuggingFace Spaces**](https://huggingface.co/spaces) for examples.
- [**LangGraph**](https://python.langchain.com/docs/langgraph/) is a framework for graph-based LLM orchestration, and is a natural next step forward for those interested in [multi-agent workflows](https://blog.langchain.dev/langgraph-multi-agent-workflows/).
- [**DSPy**](https://github.com/stanfordnlp/dspy), a flow engineering framework that allows you to optimize LLM orchestration pipelines based on empirical performance results.

<center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/DLI_Header_White.png" width="400" height="186" /></a></center>